# Tutorial: Downloading and Loading Data from Zenodo for TalentCLEF 2026

This notebook provides a step-by-step guide on downloading and loading training data for the TalentCLEF 2026 shared-task, hosted on [Zenodo](https://doi.org/10.5281/zenodo.17625261).

TalentCLEF is an initiative to advance Natural Language Processing (NLP) in Human Capital Management (HCM). It aims to create a public benchmark for model evaluation and promote collaboration to develop fair, multilingual, and flexible systems that improve Human Resources (HR) practices across different industries.

The second edition of TalentCLEF shared task’s will be part of the [Conference and Labs of the Evaluation Forum (CLEF)](https://clef2026.clef-initiative.eu/), scheduled to be held in Jena, Germany, in 2026. If you are interested in registering, you can find registration form [here](https://clef-labs-registration.dipintra.it/)


<img src="https://github.com/TalentCLEF/talentclef/blob/main/logo_talentclef.png?raw=true" alt="TalentCLEF logo" width="200"/>
<img src="https://talentclef.github.io/talentclef/docs/talentclef-2026/workshop/logo_clef_jena.svg" alt="CLEF2026 logo" width="150"/>

In this notebook you will learn how to download and load the data of the task.



## Download files

First, let's download the Task A and Task B zip files directly from Zenodo.



In [3]:
# Download
!wget https://zenodo.org/records/18449283/files/TaskA.zip
!wget https://zenodo.org/records/18449283/files/TaskB.zip
# Unzip
!unzip TaskA.zip -d taskA
!unzip TaskB.zip -d taskB

--2026-03-13 08:12:06--  https://zenodo.org/records/18449283/files/TaskA.zip
Resolving zenodo.org (zenodo.org)... 188.184.103.118, 137.138.153.219, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|188.184.103.118|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1799919 (1.7M) [application/octet-stream]
Saving to: ‘TaskA.zip’

TaskA.zip           100%[===================>]   1.72M  1.64MB/s    in 1.0s    

2026-03-13 08:12:08 (1.64 MB/s) - ‘TaskA.zip’ saved [1799919/1799919]

--2026-03-13 08:12:08--  https://zenodo.org/records/18449283/files/TaskB.zip
Resolving zenodo.org (zenodo.org)... 137.138.52.235, 137.138.153.219, 188.184.103.118, ...
Connecting to zenodo.org (zenodo.org)|137.138.52.235|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4192128 (4.0M) [application/octet-stream]
Saving to: ‘TaskB.zip’

TaskB.zip           100%[===================>]   4.00M  2.97MB/s    in 1.3s    

2026-03-13 08:12:10 (2.97 MB/s) - ‘TaskB.zip

## Task A files

### Training data



No training set is provided for this task this. However, participants are encouraged to use external resources or data from previous TalentCLEF editions if needed for their problem modeling and solution development.

### Development data

The development data for Task A is located in different folders within 'development', based on the language of the data:

In [ ]:
spanish_dev_path = "/content/taskA/TaskA/development/es"
english_dev_path = "/content/taskA/TaskA/development/en"

As described on [the data description page in TalentCLEF website](https://talentclef.github.io/talentclef/docs/talentclef-2026/data/description_corpus), the development set for each language is organized into two folders: **`queries`**, which contains the job descriptions, and **`candidates`**, which contains the résumés. In addition, a **`qrels.tsv`** file is provided, specifying the relevance judgments of the résumés for each job description.

Both the **`queries`** and **`corpus`** folders contain the documents used in the task, with each file name serving as the unique identifier for the corresponding document.

For reading those files, a function called `load_text_corpus` is created:


In [ ]:
import os
import pandas as pd

def load_text_corpus(path, id_col="c_id", encoding="utf-8"):
    """
    Load text files from a directory into a pandas DataFrame.

    Each file in the directory is treated as a document. The file name is used
    as the document identifier, and the file content is stored as text.

    Parameters
    ----------
    path : str
        Path to the directory containing the text files.
    id_col : str, optional
        Name of the column used as the document identifier
        (e.g., 'c_id', 'q_id'). Default is 'c_id'.
    encoding : str, optional
        Text encoding used to read the files. Default is 'utf-8'.

    Returns
    -------
    pd.DataFrame
        A DataFrame with two columns:
        - id_col: document identifier (file name)
        - text: document content
    """
    records = []

    for filename in os.listdir(path):
        file_path = os.path.join(path, filename)

        if os.path.isfile(file_path):
            with open(file_path, "r", encoding=encoding, errors="ignore") as f:
                text = f.read()

            records.append({
                id_col: filename,
                "text": text
            })

    return pd.DataFrame(records)

In [ ]:
en_queries = load_text_corpus(os.path.join(english_dev_path,"queries"), id_col="q_id")
en_corpus = load_text_corpus(os.path.join(english_dev_path,"corpus"), id_col="c_id")
en_qrels = pd.read_csv(os.path.join(english_dev_path,"qrels.tsv"),sep="\t", names=["query_id","iteration","corpus_id","relevancy"])

es_queries = load_text_corpus(os.path.join(spanish_dev_path,"queries"), id_col="q_id")
es_corpus = load_text_corpus(os.path.join(spanish_dev_path,"corpus"), id_col="c_id")
es_qrels = pd.read_csv(os.path.join(spanish_dev_path,"qrels.tsv"),sep="\t", names=["query_id","iteration","corpus_id","relevancy"])


Some examples of the english data:

In [ ]:
en_queries.head(2)

,q_id,text
0,44719,Cashier\n\nIntroduction\nWe are seeking a Cash...
1,35129,IELTS Teacher\n\nWe are seeking a dedicated IE...


In [ ]:
en_corpus.head(2)

,c_id,text
0,12980,"Mariana Rodríguez\nCalle Lavalleja 2847, Apart..."
1,16049,"Dmitri Volkov\nLeninsky Prospekt 42, Apartment..."


In the qrels we can explore those corpus elements that are relevant for each query.

In [ ]:
en_qrels.head(2)

,query_id,iteration,corpus_id,relevancy
0,36044,0,13884,1
1,39060,0,9516,1


For example, in the previous step, the corpus element 13884 is relevant for the query 36044

In [ ]:
from IPython.display import Markdown, display
display(Markdown("**The résumé with id 13884:**"))
print(en_corpus[en_corpus.c_id == "13884"]["text"].iloc[0][:300])

display(Markdown("**IS RELEVANT TO**"))

display(Markdown("**The job description with id 138360448:**"))
print(en_queries[en_queries.q_id == "36044"]["text"].iloc[0][:300])



**The résumé with id 13884:**

Siti Wijaya

Jalan Merdeka 42, Bandung 40173
+62-274-8156-3429 | siti.wijaya@email.com
https://www.linkedin.com/in/sitiwijaya


PROFESSIONAL SUMMARY

Results-driven Sr. Principal Product Development Engineer with 15+ years of experience leading complex semiconductor product development initiatives f


**IS RELEVANT TO**

**The job description with id 138360448:**

Failure Analysis Engineer

Introduction
We are seeking a Failure Analysis Engineer to join our team. The role focuses on investigating product and system failures, identifying root causes, and implementing corrective actions to improve reliability and quality across a range of products and processes


Some examples of the spanish data

In [ ]:
es_queries.head(2)

,q_id,text
0,86302,Profesor/a de IELTS\n\nBuscamos un/a Profesor/...
1,96027,Ingeniero/a de Análisis de Fallas\n\nIntroducc...


In [ ]:
es_corpus.head(2)

,c_id,text
0,57152,Siriporn Jantharapruk\n487 Sukhumvit Soi 12\nB...
1,63844,"Abdelrahman Osman\nKhartoum, Sudan\n+249 912 4..."


In [ ]:
es_qrels.head(2)

,query_id,iteration,corpus_id,relevancy
0,96027,0,70456,1
1,88540,0,72948,1


## Task B files

### Training

The training data for Task B is shared in 3 files:
- `job2skill.tsv`: This file has been curated to include the most representative skills for each job title in ESCO. This file contains three columns:
  - `job_id`: ESCO identifier for the job position.
  - `skill_id`: ESCO identifier for the skill.
  - `rel_type`: Indicator specifying whether the skill_id is essential or optional for a specific job_id. It can have the value “essential” or “optional.”

- `jobid2terms.json`: This JSON file contains job_id identifiers used in the training set for Task A as keys, and a list of valid lexical variants for each identifier as values.

- `skillid2terms.json`: This JSON file contains skill_id identifiers as keys, and a list of valid lexical variants for each identifier as values.



In [4]:
taskB_path = "/content/taskB/TaskB/training"

Let's read the data

In [5]:
# Read job2skill file
import pandas as pd
import os
job2skill = pd.read_csv(os.path.join(taskB_path, 'job2skill.tsv'),
                        sep="\t",
                        names=["job_id","skill_id","rel_type"])
job2skill.head()

,job_id,skill_id,rel_type
0,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/93a68dcb-3dc6...,essential
1,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/05bc7677-5a64...,essential
2,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/860be36a-d19b...,essential
3,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/fed5b267-73fa...,essential
4,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/f64fe2c2-d090...,essential


In [6]:
# Read json files
import json

with open(os.path.join(taskB_path,"jobid2terms.json"), 'r') as file:
    jobid2terms = json.load(file)

with open(os.path.join(taskB_path,"skillid2terms.json"), 'r') as file:
    skillid2terms = json.load(file)

We can map the values to the relation dataframe:

In [7]:
job2skill["job_terms"] = job2skill["job_id"].map(jobid2terms)
job2skill["skill_terms"] = job2skill["skill_id"].map(skillid2terms)

Let's see the skills for a specific job_id.



In [8]:
job_id_to_search = job2skill.job_id.to_list()[8]
job_data = job2skill[job2skill.job_id == job_id_to_search]

The job titles of the selected job_id are:

In [9]:
job_data.job_terms.iloc[0]

['technical director',
 'technical and operations director',
 'head of technical',
 'director of technical arts',
 'head of technical department',
 'technical supervisor',
 'technical manager']

For that job_id we fin 9 essential skills and 1 optional skill:

In [12]:
job_data.rel_type.value_counts()

,count
rel_type,
essential,9
optional,1


The list of skills related to the job_id are:

In [10]:
job_data.skill_terms.to_list()

[['promote health and safety',
  'promote importance of health and safety',
  'promoting health and safety',
  'advertise health and safety'],
 ['organise rehearsals',
  'organise rehearsal',
  'organize rehearsals',
  'plan rehearsals',
  'arrange rehearsals',
  'organising rehearsals',
  'schedule rehearsals'],
 ['negotiate health and safety issues with third parties',
  'agree with third parties on health and safety',
  'negotiate issues on health and safety with third parties',
  'negotiate with third parties on health and safety issues',
  'negotiate health and safety matters with third parties'],
 ['theatre techniques',
  'theatre technique',
  'theatre approaches',
  'theatre methods'],
 ['coordinate technical teams in artistic productions',
  'supervise technical teams during a production',
  'coordinate technical teams during artistic production',
  'coordinate technical teams',
  'coordinate technical teams for artistic production'],
 ['write risk assessment on performing art

In [11]:
job2skill.job_id.to_list()[8]

'http://data.europa.eu/esco/occupation/00030d09-2b3a-4efd-87cc-c4ea39d27c34'

## Dataset into Training Text

In [12]:
data = []

for _, row in job2skill.iterrows():

    job = row["job_terms"][0]        # first job term
    skill = row["skill_terms"][0]    # first skill term
    label = row["rel_type"]

    data.append({
        "job": job,
        "skill": skill,
        "label": label
    })

import pandas as pd
df = pd.DataFrame(data)

df.head()

,job,skill,label
0,technical director,promote health and safety,essential
1,technical director,organise rehearsals,essential
2,technical director,negotiate health and safety issues with third ...,essential
3,technical director,theatre techniques,essential
4,technical director,coordinate technical teams in artistic product...,essential


## Convert Labels to Numbers

In [13]:
df["label"] = df["label"].map({
    "essential": 1,
    "optional": 0
})

## Combine Job and Skill Into One Input

In [14]:

df["text"] = df["job"] + " [SEP] " + df["skill"]
df.columns
df.head()

,job,skill,label,text
0,technical director,promote health and safety,1,technical director [SEP] promote health and sa...
1,technical director,organise rehearsals,1,technical director [SEP] organise rehearsals
2,technical director,negotiate health and safety issues with third ...,1,technical director [SEP] negotiate health and ...
3,technical director,theatre techniques,1,technical director [SEP] theatre techniques
4,technical director,coordinate technical teams in artistic product...,1,technical director [SEP] coordinate technical ...


## Generate Embeddings

In [15]:
!pip install sentence-transformers scikit-learn

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(df["text"].tolist(), show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/3585 [00:00<?, ?it/s]

## Train/Test Split

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    df["label"],
    test_size=0.2,
    random_state=42
)

## Train Classifier

In [17]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)

clf.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

## Load the Validation Dataset

In [18]:
import pandas as pd
import ast

# Suppose you uploaded these files:
# corpus_elements  (no extension)
# queries          (no extension)
# qrels.tsv        (has extension)

# Load corpus elements
with open("/content/corpus_elements", "r") as f:
    corpus = pd.read_csv(f, sep="\t", header=0)
    corpus["skill_aliases"] = corpus["skill_aliases"].apply(ast.literal_eval)

# Load queries
queries = pd.read_csv("/content/queries", sep="\t", header=0)

# Load qrels
qrels = pd.read_csv("/content/qrels.tsv", sep="\t", header=None, names=["q_id","unused","c_id","rel"])

## Generate All Job-Skill Pairs

In [19]:
import ast

# Convert skill_aliases strings to list if needed
if isinstance(corpus["skill_aliases"].iloc[0], str):
    corpus["skill_aliases"] = corpus["skill_aliases"].apply(ast.literal_eval)

# Flatten aliases into a single string
corpus["text"] = corpus["skill_aliases"].apply(lambda x: " | ".join(x))

# Check
print(corpus[["c_id", "text"]].head())

import itertools

job_ids = queries["q_id"].tolist()
skill_ids = corpus["c_id"].tolist()

# Generate all job-skill pairs
pairs = list(itertools.product(job_ids, skill_ids))
print("Total job-skill pairs:", len(pairs))  # sanity check

# Create text maps
job_text_map = dict(zip(queries["q_id"], queries["jobtitle"]))
skill_text_map = dict(zip(corpus["c_id"], corpus["text"]))

# Concatenate job + skill text for classifier
pair_texts = [job_text_map[j] + " [SEP] " + skill_text_map[s] for j, s in pairs]

          c_id                                               text
0  dev_cb_sk_0  3D body scanning technologies | technologies f...
1  dev_cb_sk_1  3D modelling | 3D designing | three-dimensiona...
2  dev_cb_sk_2                                3D printing process
3  dev_cb_sk_3                                       3D texturing
4  dev_cb_sk_4  5S methodology | 5S workplace organization met...
Total job-skill pairs: 2751808


## Encode Pairs with Sentence-BERT

In [4]:
!pip install -U sentence-transformers --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 29.8 MB/s eta 0:00:00


In [20]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.preprocessing import normalize

# Load model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 1: Encode jobs and skills separately
job_texts = queries["jobtitle"].tolist()
skill_texts = [alias for aliases in corpus["skill_aliases"] for alias in aliases]

print("Encoding jobs...")
job_embeddings = model.encode(job_texts, show_progress_bar=True)
job_embeddings = normalize(job_embeddings)  # optional, for similarity

print("Encoding skills (flattened aliases)...")
skill_embeddings = model.encode(skill_texts, show_progress_bar=True)
skill_embeddings = normalize(skill_embeddings)

# Step 2: Compute quick similarity to get top-K candidates
top_k = 100  # top 100 skill candidates per job
pairs = []
pair_texts = []
job_ids = []

print("Selecting top-K candidate skills per job...")
similarity_matrix = np.dot(job_embeddings, skill_embeddings.T)  # jobs x skills

for i, job_id in enumerate(queries["q_id"]):
    top_idx = np.argsort(-similarity_matrix[i])[:top_k]  # indices of top-K skills
    for idx in top_idx:
        skill_text = skill_texts[idx]
        pair_text = job_texts[i] + " [SEP] " + skill_text
        pairs.append((job_id, idx))
        pair_texts.append(pair_text)
        job_ids.append(job_id)

print("Total candidate pairs after top-K filtering:", len(pair_texts))

# Step 3: Encode filtered pairs in batches
batch_size = 512
all_embeddings = []

for i in range(0, len(pair_texts), batch_size):
    batch_texts = pair_texts[i:i+batch_size]
    batch_emb = model.encode(batch_texts, show_progress_bar=True)
    all_embeddings.append(batch_emb)

# Combine all batches into a single array
pair_embeddings = np.vstack(all_embeddings)
print("Pair embeddings shape:", pair_embeddings.shape)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding jobs...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Encoding skills (flattened aliases)...


...

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Pair embeddings shape: (30400, 384)


In [22]:
scores = clf.predict_proba(pair_embeddings)[:, 1]

## Cosine Similarity to Generate Scores

In [42]:
import numpy as np
from sklearn.preprocessing import normalize
import pandas as pd
from collections import defaultdict

# -----------------------------
# Step 0: Prepare skill texts and mapping
# -----------------------------
skill_texts = []
skill_ids_flat = []  # maps each alias to its original skill ID

for skill_id, aliases in zip(corpus["c_id"], corpus["skill_aliases"]):
    for alias in aliases:
        skill_texts.append(alias)
        skill_ids_flat.append(skill_id)

# -----------------------------
# Step 1: Compute cosine similarities between job and skill embeddings
# -----------------------------
job_embeddings = normalize(job_embeddings)      # normalize if not already done
skill_embeddings = normalize(skill_embeddings)

similarity_matrix = np.dot(job_embeddings, skill_embeddings.T)  # jobs x skills

# -----------------------------
# Step 2: Select top-K candidate skills per job
# -----------------------------
top_k = 100
pairs = []       # (job_id, skill_flat_idx)
pair_texts = []  # job + skill text pairs
job_ids = []

for i, job_id in enumerate(queries["q_id"]):
    top_idx = np.argsort(-similarity_matrix[i])[:top_k]
    for idx in top_idx:
        pair_texts.append(queries["jobtitle"].iloc[i] + " [SEP] " + skill_texts[idx])
        pairs.append((job_id, idx))
        job_ids.append(job_id)

print("Total candidate pairs after top-K filtering:", len(pair_texts))

# -----------------------------
# Step 3: Generate scores using cosine similarity directly
# -----------------------------
# score = similarity from matrix
scores = []
for job_id, skill_idx in pairs:
    scores.append(similarity_matrix[queries["q_id"].tolist().index(job_id), skill_idx])
scores = np.array(scores)

# -----------------------------
# Step 4: Load valid job and skill IDs from qrels
# -----------------------------
qrels_df = pd.read_csv("/content/qrels.tsv", sep="\t", header=None, names=["q_id","iter","doc_id","rel"])
valid_job_ids = set(qrels_df.q_id)
valid_skill_ids = set(qrels_df.doc_id)

# -----------------------------
# Step 5: Build results dictionary
# -----------------------------
results = defaultdict(list)
for (job_id, skill_idx), score in zip(pairs, scores):
    skill_id = skill_ids_flat[skill_idx]  # correct mapping
    if job_id in valid_job_ids and skill_id in valid_skill_ids:
        results[job_id].append((skill_id, score))

# Sort each job's skills by descending score
for job_id in results:
    results[job_id] = sorted(results[job_id], key=lambda x: -x[1])

# -----------------------------
# Step 6: Convert to TREC-style run file
# -----------------------------
run_rows = []
for job_id, skill_list in results.items():
    for rank, (skill_id, score) in enumerate(skill_list, start=1):
        run_rows.append([job_id, "Q0", skill_id, rank, score])

run_df = pd.DataFrame(run_rows, columns=["q_id", "Q0", "doc_id", "rank", "score"])
run_file_path = "/content/taskB_run.tsv"
run_df.to_csv(run_file_path, sep="\t", index=False, header=False)

print("Run file saved at:", run_file_path)

Total candidate pairs after top-K filtering: 30400
Run file saved at: /content/taskB_run.tsv


## Eavluate

In [43]:

!pip install ranx --quiet
!pip install ranx --quiet

!python talentclef_evaluate.py \
    --task B \
    --qrels /content/qrels.tsv \
    --run /content/taskB_run.tsv

TalentCLEF 2026 - Task B Evaluation
Received parameters:
  Task: B
  Qrels: /content/qrels.tsv
  Run: /content/taskB_run.tsv

Loading qrels...
Loading run...

Running Task B evaluation...

EVALUATION RESULTS

--- General Relevance (1 or 2 → relevant) ---
ndcg: 0.0995
map: 0.0256
mrr: 0.5287
precision@5: 0.2921
precision@10: 0.2944
precision@100: 0.1127

--- Graded Relevance (2 = core skill, 1 = contextual skill) ---
ndcg: 0.1014
map: 0.0256
mrr: 0.5287
precision@5: 0.2921
precision@10: 0.2944
precision@100: 0.1127
